In [0]:
import dlt
import pyspark.sql.functions as F

***1.Ingest the data into data lakehouse - bronze_addresses***

In [0]:
@dlt.table(
    name = "bronze_addresses",
    table_properties = {"quality":"bronze"},
    comment = "Raw addresses data ingested from the source system"
)
def create_bronze_addresses():
    return(
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.Format", "csv")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/Volumes/circuitbox_sivan/landing/operational_data/addresses/")
        
        .select(
            "*",
            F.col("_metadata.file_path").alias("input_file_path"),
            F.current_timestamp().alias("ingest_timestamp")
        )
    )

****2. Perfrom data quality checks and***

In [0]:
@dlt.table(
    name = "silver_addresses_cleaned",
    table_properties = {"quality":"silver"},
    comment = "Cleaned addresses data"
)
@dlt.expect_or_fail("valid_customer_id", "customer_id is not null")
@dlt.expect_or_drop("valid_address", "address_line_1 is not null")
@dlt.expect("valid_postcode", "LENGTH(postcode) = 5")
def create_silver_addresses_cleaned():
    return(
        spark.readStream.table("live.bronze_addresses")
        .select(
            "customer_id",
            "address_line_1",
            "city",
            "state",
            "postcode",
            F.col("created_date").cast("date").alias("created_date")
        )
    )

***3.Apply changes to the Addresses date (SCD Type 2) - Silver_addresses***

In [0]:
dlt.create_streaming_table(
    name = "silver_addresses",
    table_properties = {"quality":"silver"},
    comment = "SCD type 2 dimension address data"
)
dlt.apply_changes(
    target = "silver_addresses",
    source = "live.silver_addresses_cleaned",
    keys = ["customer_id"],
    sequence_by = F.col("created_date"),
    stored_as_scd_type = 2
)